# Portfolio Returns & Realized Vol — Example

Fetch aligned daily returns for a weighted portfolio from yfinance (via the `portfolio_vol` module in this folder), then compute realized portfolio vol over the full fetched sample.

The vol math matches the VBA reference: population covariance (divide by n, `ddof=0`). To change the window, just change `START`/`END` in the config cell.

## 1. One-time setup

Uncomment and run the next cell once to install dependencies into the active Python env. After that you can re-run the notebook without it.

In [11]:
 %pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\kpa32\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 2. Define the portfolio & window

- `WEIGHTS` — dict of ticker → weight. Case-insensitive; must sum to 1.0 (or pass `normalize=True`).
- `START`/`END` — yfinance fetch window. All downstream math runs on the full fetched sample.

In [39]:
import numpy as np
import pandas as pd

from portfolio_vol import Portfolio, fetch_returns

WEIGHTS = {
    "SPY": 0.40,
    "QQQ": 0.30,
    "USO": 0.10,
    "TLT": 0.10,
    "GLD": 0.10,
}

START = "2026-01-01"
END   = "2026-04-01"

TRADING_DAYS_PER_YEAR = 252  # annualization factor

portfolio = Portfolio.from_weights(WEIGHTS)
portfolio.weights

GLD    0.1
QQQ    0.3
SPY    0.4
TLT    0.1
USO    0.1
dtype: float64

## 3. Fetch daily returns

Log returns, aligned on the intersection of trading days across tickers.

In [40]:
returns = fetch_returns(portfolio, start=START, end=END)
print(f"shape: {returns.shape}   range: {returns.index.min().date()} → {returns.index.max().date()}")
returns.tail()

shape: (60, 5)   range: 2026-01-05 → 2026-03-31


Ticker,GLD,QQQ,SPY,TLT,USO
date,,,,,
2026-03-25,0.029646,0.006554,0.005557,0.009604,-0.010091
2026-03-26,-0.038319,-0.024157,-0.018020,-0.008442,0.033561
2026-03-27,0.034492,-0.019730,-0.017199,-0.005473,0.057499
2026-03-30,-0.000289,-0.007673,-0.003349,0.013224,0.044333
2026-03-31,0.037193,0.033293,0.028653,-0.001038,-0.020072


## 4. Realized portfolio vol

Matches the Excel/VBA calc over the full sample:
1. Covariance matrix Σ with `ddof=0` (population, divide by n — same as VBA `WorksheetFunction.Covar`).
2. Variance = wᵀ Σ w, vol = √variance, annualized vol = daily_vol × √252.

In [41]:
cov = returns.cov(ddof=0)                # population covariance, matches VBA Covar
w   = portfolio.weights.loc[cov.columns]  # reindex so rows/cols/weights align

# Step 1: Σw  (the "intermediate result" column in the Excel sheet)
sigma_w = cov @ w

# Step 2: wᵀ Σ w  (scalar variance)
daily_var = float(w @ sigma_w)

# Step 3: sqrt
daily_vol = np.sqrt(daily_var)
ann_vol   = daily_vol * np.sqrt(TRADING_DAYS_PER_YEAR)

print(f"sample: {len(returns)} days  ({returns.index.min().date()} → {returns.index.max().date()})")
print(f"daily portfolio vol:      {daily_vol:.4%}")
print(f"annualized portfolio vol: {ann_vol:.2%}")

sample: 60 days  (2026-01-05 → 2026-03-31)
daily portfolio vol:      0.7892%
annualized portfolio vol: 12.53%


In [42]:
# Sanity check: same number via the portfolio return series.
# Uses population stdev (ddof=0) to stay consistent with the matrix path.
port_ret = returns @ w
daily_vol_check = port_ret.std(ddof=0)
print(f"via port_ret.std(ddof=0): {daily_vol_check:.6%}")
print(f"via wᵀ Σ w:               {daily_vol:.6%}")
assert np.isclose(daily_vol, daily_vol_check), "vol paths disagree"

via port_ret.std(ddof=0): 0.789168%
via wᵀ Σ w:               0.789168%


## 5. Component vol & correlation matrix

Same full sample. Weighted-average component vol is always ≥ portfolio vol — their ratio is a crude diversification measure.

In [43]:
# Per-name daily vol (sqrt of the diagonal of the cov matrix)
component_daily_vol = pd.Series(np.sqrt(np.diag(cov)), index=cov.columns, name="daily_vol")
component_ann_vol   = component_daily_vol * np.sqrt(TRADING_DAYS_PER_YEAR)

# Weighted-average component vol (ignores correlation)
wavg_daily_vol = float((w * component_daily_vol).sum())
wavg_ann_vol   = wavg_daily_vol * np.sqrt(TRADING_DAYS_PER_YEAR)

# Mean daily return by ticker (aligned with weights)
mean_returns = returns[cov.columns].mean()
asset_contrib_return = w * mean_returns

# Per-asset full-period total returns and weighted contribution
asset_total_return = (1.0 + returns[cov.columns]).prod() - 1.0
asset_total_contribution = w * asset_total_return

# Portfolio expected daily return
portfolio_return = float(asset_contrib_return.sum())

# Full-period cumulative return from daily portfolio returns (daily rebalanced)
port_ret = returns @ w
full_period_return = float((1.0 + port_ret).prod() - 1.0)

# Weighted total return implied by asset-level totals (buy-and-hold style)
weighted_asset_total_return = float(asset_total_contribution.sum())

# Portfolio sharpe ratio with vol scaled to trading days in sample
sharpe_ratio = portfolio_return / daily_vol * np.sqrt(len(returns))

print(f"portfolio return (mean daily): {portfolio_return:.2%}")
print(f"portfolio return (full period, daily rebalanced): {full_period_return:.2%}")
print(f"portfolio return (full period, weighted asset totals): {weighted_asset_total_return:.2%}")
print(f"portfolio sharpe ratio (daily return / daily vol, scaled to sample length): {sharpe_ratio:.3f}")
print(f"weighted-avg component vol — daily: {wavg_daily_vol:.2%}   annualized: {wavg_ann_vol:.2%}")
print(f"portfolio vol              — daily: {daily_vol:.2%}   annualized: {ann_vol:.2%}")
print(f"diversification ratio (wavg / port): {wavg_daily_vol / daily_vol:.3f}")

# Compute avg correlation
avg_corr = (ann_vol**2) / wavg_ann_vol**2
print(f"average correlation: {avg_corr:.3f}")

output_table = pd.DataFrame(
    {
        "weight": w,
        "asset_mean_daily_return": mean_returns,
        "daily_return_contribution": asset_contrib_return,
        "asset_total_return": asset_total_return,
        "total_return_contribution": asset_total_contribution,
        "total_contribution_share": asset_total_contribution / weighted_asset_total_return,
        "daily_vol": component_daily_vol,
        "ann_vol": component_ann_vol,
    }
)

output_table.style.format(
    {
        "weight": "{:.2%}",
        "asset_mean_daily_return": "{:.2%}",
        "daily_return_contribution": "{:.2%}",
        "asset_total_return": "{:.2%}",
        "total_return_contribution": "{:.2%}",
        "total_contribution_share": "{:.2%}",
        "daily_vol": "{:.2%}",
        "ann_vol": "{:.2%}",
    }
)

portfolio return (mean daily): 0.05%
portfolio return (full period, daily rebalanced): 3.16%
portfolio return (full period, weighted asset totals): 4.64%
portfolio sharpe ratio (daily return / daily vol, scaled to sample length): 0.539
weighted-avg component vol — daily: 1.37%   annualized: 21.83%
portfolio vol              — daily: 0.79%   annualized: 12.53%
diversification ratio (wavg / port): 1.742
average correlation: 0.329


,weight,asset_mean_daily_return,daily_return_contribution,asset_total_return,total_return_contribution,total_contribution_share,daily_vol,ann_vol
Ticker,,,,,,,,
GLD,10.00%,0.13%,0.01%,5.69%,0.57%,12.26%,2.68%,42.49%
QQQ,30.00%,-0.10%,-0.03%,-6.12%,-1.84%,-39.58%,1.16%,18.37%
SPY,40.00%,-0.08%,-0.03%,-4.78%,-1.91%,-41.18%,0.90%,14.29%
TLT,10.00%,0.01%,0.00%,0.19%,0.02%,0.41%,0.66%,10.51%
USO,10.00%,1.02%,0.10%,78.03%,7.80%,168.09%,3.34%,52.99%


In [44]:
# Correlation matrix (invariant to ddof, so this matches either convention)
returns.corr()

Ticker,GLD,QQQ,SPY,TLT,USO
Ticker,,,,,
GLD,1.000000,0.295922,0.254041,0.040979,0.167888
QQQ,0.295922,1.000000,0.962125,0.206269,-0.392466
SPY,0.254041,0.962125,1.000000,0.248596,-0.429733
TLT,0.040979,0.206269,0.248596,1.000000,-0.403181
USO,0.167888,-0.392466,-0.429733,-0.403181,1.000000


## 6. Error handling — what to expect

- `PortfolioError` — bad spec (weights don't sum to 1, duplicate ticker, non-finite weight, empty dict).
- `FetchError` — yfinance issue (bad ticker, no overlap across names, all retries failed, too few observations after alignment).

Try swapping in a bogus ticker below to see the error path.

In [45]:
from portfolio_vol import FetchError, PortfolioError

try:
    bad = Portfolio.from_weights({"SPY": 0.5, "NOTATICKER": 0.5})
    fetch_returns(bad, start=START, end=END)
except (FetchError, PortfolioError) as e:
    print(f"{type(e).__name__}: {e}")

$NOTATICKER: possibly delisted; no timezone found

1 Failed download:
['NOTATICKER']: possibly delisted; no timezone found


FetchError: yfinance returned no data for: ['NOTATICKER']. Check the ticker symbols and date range.
